# Lab 03 -- What RAG Is Made Of

Week 5 (content: Context Engineering, week 4) · Narxoz University

A retrieval-augmented generation system is four small, separately-testable pieces: **chunking** (cut documents into retrievable units), **embedding** (turn text into vectors), **retrieval** (find the vectors closest to a question), and **generation** (answer using only what retrieval found). This lab builds all four by hand, on four short policy articles from a fictional bank -- same domain as Lab 01's support queue.

**No API key. No installation. No cost.** Two small open models are downloaded inside this notebook: an embedding model for retrieval, and an instruction-following model for generation.

## Setup

Open this notebook in Google Colab and run the next cell first -- it installs the one package Colab does not ship with. Runtime → Run all is fine for this cell only; from Part 1 onward, run cells one at a time so you can write your prediction before you see the answer.

In [1]:
!pip install -q sentence-transformers

In [2]:
import torch
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
from transformers import AutoModelForCausalLM, AutoTokenizer

## Part 0 -- the corpus

Four articles from the bank's policy pages, all about payment cards -- deliberately near-neighbours, so a retrieval system has to tell them apart, not just notice the word "card". Each article is a list of sentences; `needle_idx` marks the one sentence that actually answers that article's two questions (`literal`, a direct restatement, and `paraphrase`, the same need in different words).

Read all four before running the next cell.

In [3]:
CORPUS = [
    {
        "id": "cards-lost",
        "needle_idx": 5,
        "sentences": [
            "If your card is lost or stolen, block it at once in the mobile app under Cards, then Block, or call the round-the-clock hotline printed on the back of the card.",
            "Blocking takes effect within 60 seconds, after which the card cannot be used for payments or cash withdrawals.",
            "You are not liable for any transaction made after you report the loss.",
            "Earlier transactions are reviewed one by one, and the Bank refunds those you did not authorise, provided you never shared your PIN.",
            "A replacement card is issued free of charge and is ready at your chosen branch within 5 working days.",
            "If you need a card the same day, the branch can issue an instant card for 3000 tenge.",
        ],
        "queries": {
            "literal": "How much does an instant replacement card cost at the branch?",
            "paraphrase": "I lost my wallet this morning and need a working card by tonight - what will it cost me?",
        },
    },
    {
        "id": "cards-pin-block",
        "needle_idx": 3,
        "sentences": [
            "After three wrong PIN entries in a row, the card is blocked automatically to protect your money.",
            "A card blocked this way is not lost: your funds and your account are untouched.",
            "To unblock it, open the app, choose the card, and confirm with the six-digit code we send by SMS.",
            "Unblocking in the app is free and takes effect within 2 minutes.",
            "If you have no access to the app, any branch can unblock the card on presentation of your ID, also free of charge.",
            "After unblocking, the wrong-attempt counter resets to zero, and you have three new attempts.",
        ],
        "queries": {
            "literal": "How long does it take for the card to work again after I unblock it in the app?",
            "paraphrase": "I mistyped my code at the till and the card stopped working - once I sort it out on my phone, how soon can I pay with it again?",
        },
    },
    {
        "id": "cards-expiry",
        "needle_idx": 4,
        "sentences": [
            "Your card is valid until the last day of the month printed on its face.",
            "We start reissuing it automatically 30 days before that date, so you do not need to apply.",
            "The new card keeps the same number and PIN, but has a new expiry date and a new security code.",
            "It is delivered to your home branch, and we notify you by SMS when it is ready.",
            "Uncollected cards are destroyed after 60 days for security reasons, and a new one then costs 1500 tenge.",
        ],
        "queries": {
            "literal": "How many days do I have to collect a reissued card before it is destroyed?",
            "paraphrase": "My new card has been waiting at the branch for ages and I keep forgetting to go - is there a point where they throw it away?",
        },
    },
    {
        "id": "cards-contactless",
        "needle_idx": 0,
        "sentences": [
            "Contactless payments up to 15000 tenge do not require a PIN.",
            "After five contactless payments in a row without a PIN, the terminal asks for the PIN once, and the counter starts again.",
            "You can lower the limit, or switch contactless off completely, in the app under Card settings.",
            "Changes to the limit take effect immediately and are free.",
            "Payments above the limit always require a PIN, whether or not you have changed the setting.",
        ],
        "queries": {
            "literal": "What is the maximum contactless payment that does not need a PIN?",
            "paraphrase": "Up to what sum can I simply hold my card near the reader and walk away without typing anything?",
        },
    },
]

for article in CORPUS:
    print(f"{article['id']}: {len(article['sentences'])} sentences, needle is #{article['needle_idx']}")

cards-lost: 6 sentences, needle is #5
cards-pin-block: 6 sentences, needle is #3
cards-expiry: 5 sentences, needle is #4
cards-contactless: 5 sentences, needle is #0


## Part 1 -- chunking

A *chunk* is the unit retrieval works on. Two ways to cut these four articles into chunks:

- **article-level**: one chunk per article (its sentences joined into one block).
- **sentence-level**: one chunk per sentence.

✏️ **Predict, before running the next cell**: from the sentence counts printed in Part 0, how many chunks will each strategy produce in total?

### *Prediction: I expect 4 article-level chunks and 22 sentence-level chunks.*

In [5]:
def chunk_by_article(corpus):
    return [
        {"id": a["id"], "article_id": a["id"], "text": " ".join(a["sentences"])}
        for a in corpus
    ]


def chunk_by_sentence(corpus):
    return [
        {"id": f"{a['id']}:{i}", "article_id": a["id"], "text": s}
        for a in corpus
        for i, s in enumerate(a["sentences"])
    ]


article_chunks = chunk_by_article(CORPUS)
sentence_chunks = chunk_by_sentence(CORPUS)
print(f"article-level chunks: {len(article_chunks)}")
print(f"sentence-level chunks: {len(sentence_chunks)}")

article-level chunks: 4
sentence-level chunks: 22


## Part 2 -- embedding and retrieval

The embedding model turns text into a vector; cosine similarity measures how close two vectors point. `multilingual-e5-small` (the model below) was trained with a convention: prefix every *document* with `"passage: "` and every *question* with `"query: "` -- without the prefixes, retrieval quality drops noticeably. This is a property of this specific model family, not a universal rule.

✏️ **Predict, before running**: for the `cards-lost` article's `literal` query (`"How much does an instant replacement card cost at the branch?"`), which of the four articles do you expect **article-level** retrieval to return? And which exact **sentence** (by its content, not its number) do you expect **sentence-level** retrieval to return? Read the four articles in Part 0 to answer -- don't guess blind.

### *Prediction: Article-level retrieval will return cards-lost. Sentence-level retrieval will return the sentence about issuing an instant card for 3000 tenge.*



In [6]:
embedder = SentenceTransformer("intfloat/multilingual-e5-small")

article_vecs = embedder.encode(
    [f"passage: {c['text']}" for c in article_chunks], convert_to_tensor=True
)
sentence_vecs = embedder.encode(
    [f"passage: {c['text']}" for c in sentence_chunks], convert_to_tensor=True
)


def retrieve(chunks, chunk_vecs, query, k=3):
    q_vec = embedder.encode(f"query: {query}", convert_to_tensor=True)
    scores = cos_sim(q_vec, chunk_vecs)[0]
    top = torch.topk(scores, k=min(k, len(chunks)))
    return [(chunks[i], float(top.values[j])) for j, i in enumerate(top.indices)]

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [7]:
demo_article = CORPUS[0]  # cards-lost
query = demo_article["queries"]["literal"]
print(f"query: {query}\n")

top_article = retrieve(article_chunks, article_vecs, query, k=1)[0]
print(f"article-level retrieved: {top_article[0]['article_id']} (score {top_article[1]:.3f})\n")

top_sentences = retrieve(sentence_chunks, sentence_vecs, query, k=3)
for chunk, score in top_sentences:
    print(f"sentence-level [{score:.3f}] {chunk['id']}: {chunk['text']}")

query: How much does an instant replacement card cost at the branch?

article-level retrieved: cards-lost (score 0.869)

sentence-level [0.851] cards-lost:4: A replacement card is issued free of charge and is ready at your chosen branch within 5 working days.
sentence-level [0.848] cards-lost:5: If you need a card the same day, the branch can issue an instant card for 3000 tenge.
sentence-level [0.828] cards-expiry:4: Uncollected cards are destroyed after 60 days for security reasons, and a new one then costs 1500 tenge.


Now the harder version: the same article's **paraphrase** query -- the same need, described without reusing the article's own words.

✏️ **Predict again before running**: do you expect the paraphrase to change either answer above?

### *Prediction: The paraphrase may change the similarity scores and may change the sentence-level ranking, but I expect article-level retrieval to return cards-lost.*

In [8]:
query = demo_article["queries"]["paraphrase"]
print(f"query: {query}\n")

top_article = retrieve(article_chunks, article_vecs, query, k=1)[0]
print(f"article-level retrieved: {top_article[0]['article_id']} (score {top_article[1]:.3f})\n")

top_sentences = retrieve(sentence_chunks, sentence_vecs, query, k=3)
for chunk, score in top_sentences:
    print(f"sentence-level [{score:.3f}] {chunk['id']}: {chunk['text']}")

query: I lost my wallet this morning and need a working card by tonight - what will it cost me?

article-level retrieved: cards-expiry (score 0.881)

sentence-level [0.864] cards-expiry:4: Uncollected cards are destroyed after 60 days for security reasons, and a new one then costs 1500 tenge.
sentence-level [0.861] cards-lost:4: A replacement card is issued free of charge and is ready at your chosen branch within 5 working days.
sentence-level [0.860] cards-pin-block:1: A card blocked this way is not lost: your funds and your account are untouched.


## Part 3 -- generation, without and with retrieval

`Qwen2.5-1.5B-Instruct` has never seen this fictional bank's policy -- it cannot know the real answer, only produce a fluent-sounding guess. Retrieval's job is to hand it the one sentence it needs so it does not have to guess.

✏️ **Predict, before running the next two cells**: will the no-context answer refuse to guess, or will it state a specific but made-up number?

### *Prediction: Without retrieved context, the model may give a specific but incorrect answer because it does not know the fictional bank's policy.*

In [9]:
GEN_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL)
gen_model = AutoModelForCausalLM.from_pretrained(GEN_MODEL, dtype=torch.float32)
gen_model.eval()


def generate(prompt, max_new_tokens=90):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt")
    with torch.no_grad():
        output_ids = gen_model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [10]:
query = demo_article["queries"]["literal"]

no_context_answer = generate(f"Answer in one or two sentences: {query}")
print("--- WITHOUT retrieved context ---")
print(no_context_answer)

--- WITHOUT retrieved context ---
I'm sorry, but I cannot provide you with that information as it may violate our terms of service to answer questions about specific products or services. However, you can usually find this information on the official website of the company or contact their customer support for assistance.


In [11]:
needle = demo_article["sentences"][demo_article["needle_idx"]]

with_context_prompt = (
    "Answer the question using ONLY the fact below. "
    "If the fact does not answer the question, say you don't know.\n\n"
    f"Fact: {needle}\n\nQuestion: {query}"
)
with_context_answer = generate(with_context_prompt)
print("--- WITH retrieved context ---")
print(with_context_answer)

--- WITH retrieved context ---
The cost of an instant replacement card at the branch is 3000 tenge.


## What you hand in

One page, submitted to Canvas as this notebook (*File → Download → .ipynb*) after you have run every cell and answered every ✏️ prompt above.

1. Your Part 1 prediction and the two measured chunk counts.
2. Your Part 2 predictions (article-level and sentence-level, both queries) next to what actually came back.
3. The two Part 3 answers (without and with context), and one sentence on what the no-context answer got wrong.
4. One sentence: for a corpus like this one, would you chunk by article or by sentence, and why?

## *For this corpus, I would choose sentence-level chunking when precise facts are needed, because each chunk contains a focused statement. However, sentence-level retrieval can fail when similar sentences have different meanings, so retrieval quality must be evaluated.*

### *Part 2 - Analysis*

For the literal query, article-level retrieval correctly returned `cards-lost` with a score of 0.869. Sentence-level retrieval ranked the free replacement sentence first (0.851), while the correct sentence about the instant card costing 3000 tenge ranked second (0.848).

For the paraphrase query, article-level retrieval returned `cards-expiry` (0.881), which was incorrect. Sentence-level retrieval also ranked a sentence from `cards-expiry` first (0.864). This shows that paraphrasing can make retrieval less reliable.

### *Part 3 - Analysis*

Without retrieved context, the model refused to answer and did not provide the requested price. With the correct context, it answered that an instant replacement card costs 3000 tenge.

For this corpus, sentence-level chunking can provide focused facts, but similar wording can cause incorrect rankings. Article-level chunking preserves more context, although it can also retrieve the wrong article for a paraphrased query.

## GitHub Core 1

In [12]:
# Core 1 — Repeat retrieval for cards-pin-block

demo_article_pin = CORPUS[1]
query_pin = demo_article_pin["queries"]["literal"]

print(f"Query: {query_pin}\n")

top_article_pin = retrieve(article_chunks, article_vecs, query_pin, k=1)[0]
print(
    f"Article-level retrieved: {top_article_pin[0]['article_id']} "
    f"(score {top_article_pin[1]:.3f})\n"
)

top_sentences_pin = retrieve(sentence_chunks, sentence_vecs, query_pin, k=3)
for chunk, score in top_sentences_pin:
    print(f"Sentence-level [{score:.3f}] {chunk['id']}: {chunk['text']}")


Query: How long does it take for the card to work again after I unblock it in the app?

Article-level retrieved: cards-pin-block (score 0.908)

Sentence-level [0.887] cards-lost:1: Blocking takes effect within 60 seconds, after which the card cannot be used for payments or cash withdrawals.
Sentence-level [0.879] cards-lost:0: If your card is lost or stolen, block it at once in the mobile app under Cards, then Block, or call the round-the-clock hotline printed on the back of the card.
Sentence-level [0.878] cards-pin-block:2: To unblock it, open the app, choose the card, and confirm with the six-digit code we send by SMS.


## GitHub Core 2: Recall@1

In [13]:

# Core 2 - Recall@1 across all eight queries

article_correct = 0
sentence_correct = 0
total_queries = 0

for article in CORPUS:
    for query_type in ["literal", "paraphrase"]:
        query = article["queries"][query_type]
        total_queries += 1

        # Article-level: did we retrieve the correct article?
        top_article = retrieve(article_chunks, article_vecs, query, k=1)[0][0]
        article_hit = top_article["article_id"] == article["id"]
        article_correct += int(article_hit)

        # Sentence-level: did we retrieve the exact needle sentence?
        top_sentence = retrieve(sentence_chunks, sentence_vecs, query, k=1)[0][0]
        expected_needle_id = f"{article['id']}:{article['needle_idx']}"
        sentence_hit = top_sentence["id"] == expected_needle_id
        sentence_correct += int(sentence_hit)

        print(
            f"{article['id']} | {query_type} | "
            f"article hit: {article_hit} | "
            f"sentence hit: {sentence_hit}"
        )

print("\n--- Recall@1 results ---")
print(
    f"Article-level Recall@1: {article_correct}/{total_queries} "
    f"= {article_correct / total_queries:.3f}"
)
print(
    f"Sentence-level Recall@1: {sentence_correct}/{total_queries} "
    f"= {sentence_correct / total_queries:.3f}"
)


cards-lost | literal | article hit: True | sentence hit: False
cards-lost | paraphrase | article hit: False | sentence hit: False
cards-pin-block | literal | article hit: True | sentence hit: False
cards-pin-block | paraphrase | article hit: False | sentence hit: False
cards-expiry | literal | article hit: True | sentence hit: True
cards-expiry | paraphrase | article hit: True | sentence hit: True
cards-contactless | literal | article hit: True | sentence hit: True
cards-contactless | paraphrase | article hit: True | sentence hit: False

--- Recall@1 results ---
Article-level Recall@1: 6/8 = 0.750
Sentence-level Recall@1: 3/8 = 0.375


## GitHub Core 3: Near-tie

In [14]:
# Core 3 — Measure the near-tie

query_lost = CORPUS[0]["queries"]["literal"]
top3_lost = retrieve(sentence_chunks, sentence_vecs, query_lost, k=3)

needle_id = f"{CORPUS[0]['id']}:{CORPUS[0]['needle_idx']}"
correct_result = next(
    (item for item in top3_lost if item[0]["id"] == needle_id),
    None
)

for chunk, score in top3_lost:
    print(f"{score:.4f} | {chunk['id']} | {chunk['text']}")

if correct_result is not None:
    wrong_score = top3_lost[0][1]
    correct_score = correct_result[1]
    gap = wrong_score - correct_score

    print(f"\nTop-1 score: {wrong_score:.4f}")
    print(f"Correct needle score: {correct_score:.4f}")
    print(f"Score gap: {gap:.4f}")
else:
    print("\nThe correct needle is not in the top 3.")


0.8514 | cards-lost:4 | A replacement card is issued free of charge and is ready at your chosen branch within 5 working days.
0.8476 | cards-lost:5 | If you need a card the same day, the branch can issue an instant card for 3000 tenge.
0.8277 | cards-expiry:4 | Uncollected cards are destroyed after 60 days for security reasons, and a new one then costs 1500 tenge.

Top-1 score: 0.8514
Correct needle score: 0.8476
Score gap: 0.0038


## Advanced: Generation in Russian

In [15]:
# Advanced 5 — Generation in Russian

query_ru = "Сколько стоит моментальная карта в отделении?"

# Russian translation of the relevant fact.
needle_ru = (
    "Если вам нужна карта в тот же день, "
    "отделение может выпустить моментальную карту за 3000 тенге."
)

prompt_ru = (
    "Ответь на вопрос только на основе приведённого факта. "
    "Если факта недостаточно, скажи, что не знаешь.\n\n"
    f"Факт: {needle_ru}\n\n"
    f"Вопрос: {query_ru}"
)

answer_ru = generate(prompt_ru)
print("Question:", query_ru)
print("Answer:", answer_ru)


Question: Сколько стоит моментальная карта в отделении?
Answer: Стоит моментальная карта в отделении 3000 тенге.


**Advanced 5 — Analysis**

The model answered the question in Russian and correctly returned the price: 3000 tenge. The answer was concise and consistent with the provided context.

**Translation quality check**

I could not complete the translation quality check because `corpus.py` was not available in the lab repository. The assignment notes that the Russian and Kazakh translations have not been reviewed by a native speaker, so their overall quality cannot be confirmed from the answer to this task alone.